# Medical Records RAG Chatbot

This notebook builds a fully local, CPU-friendly **Retrieval-Augmented Generation (RAG)** chatbot over a folder of patient medical records. No API keys. No cloud. Everything runs on your machine.

---
![RAG Process](https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSi-YU4IFJ1LMA5Dk_0Tc46EcQfRvZSyfZKsg&s)
## How It Works

RAG is a two-stage architecture:

1. **Retrieval** — When you ask a question, the system searches your document corpus using semantic similarity (via embeddings) to find the most relevant passages.
2. **Generation** — Those passages are injected as context into a language model prompt, which then synthesizes a grounded answer.

This means the LLM answers strictly from the documents — not from its general training knowledge — making it accurate and auditable.


---

## 📁 Data Format

Place plain `.txt` files inside the `patient_records/` folder (already created). Each file represents one patient and may contain multiple visit dates, discussions, and prescribed medications. The RAG system ingests **all files** from that folder automatically.


## Step 1 — Install Dependencies

We install everything needed in one shot. The key packages:

- **`llama-cpp-python`** — Python bindings for `llama.cpp`, which lets us run quantized GGUF models on CPU with no GPU required.
- **`sentence-transformers`** — Generates dense vector embeddings from text, used for semantic search.
- **`faiss-cpu`** — Facebook's efficient similarity search library (CPU version).
- **`huggingface-hub`** — Provides easy model download utilities from Hugging Face.

> ⚠️ `llama-cpp-python` compiles C++ extensions during install. This may take **2–5 minutes** the first time.

In [ ]:
import subprocess, sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "llama-cpp-python",
    "sentence-transformers",
    "faiss-cpu",
    "huggingface-hub"
])

print("All dependencies installed.")

All dependencies installed.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2 — Download the Quantized LLM

We use **Gemma 2 2B Instruct** in **Q4_K_M GGUF** format from Hugging Face.

- **Gemma 2 2B** is Google's open-weight small model — surprisingly capable for its size.
- **Q4_K_M** is a 4-bit quantization: the model weights are compressed from 32-bit floats down to ~4 bits each, reducing memory from ~8 GB → ~1.5 GB with minimal quality loss.
- **GGUF** is the file format used by `llama.cpp` for optimized CPU inference.

The model is downloaded once and cached locally. Subsequent runs skip the download.

In [ ]:
from huggingface_hub import hf_hub_download
import os

MODEL_REPO = "bartowski/gemma-2-2b-it-GGUF"
MODEL_FILE = "gemma-2-2b-it-Q4_K_M.gguf"
MODEL_DIR  = "./models"

os.makedirs(MODEL_DIR, exist_ok=True)

model_path = os.path.join(MODEL_DIR, MODEL_FILE)

if not os.path.exists(model_path):
    print("📥 Downloading model — this may take a few minutes...")
    model_path = hf_hub_download(
        repo_id=MODEL_REPO,
        filename=MODEL_FILE,
        local_dir=MODEL_DIR
    )
    print(f"Model saved to: {model_path}")
else:
    print(f"Model already cached at: {model_path}")

## Step 3 — Load the LLM

We initialize `llama.cpp` with the downloaded model. Key parameters:

| Parameter | Value | Purpose |
|-----------|-------|---------|
| `n_ctx` | 4096 | Context window — how many tokens the model sees at once |
| `n_threads` | CPU count | Parallelizes inference across all available CPU cores |
| `n_gpu_layers` | 0 | Forces fully CPU-only inference (set higher if you have a GPU) |
| `verbose` | False | Suppresses low-level llama.cpp logs |

Loading takes **10–30 seconds** depending on your machine.

In [ ]:
from llama_cpp import Llama
import os

print("Loading LLM into memory...")

llm = Llama(
    model_path=model_path,
    n_ctx=4096,
    n_threads=os.cpu_count(),
    n_gpu_layers=0,
    verbose=False
)

print("LLM loaded and ready.")

## Step 4 — Load the Embedding Model

The embedding model converts text into high-dimensional vectors (numbers) that encode **semantic meaning**. Sentences with similar meaning end up close together in this vector space.

We use **`all-MiniLM-L6-v2`** — a 22M parameter model that:
- Runs entirely on CPU in milliseconds
- Produces 384-dimensional vectors
- Is one of the best embeddings per compute-dollar available

This is the same model used in many production RAG pipelines.

In [ ]:
from sentence_transformers import SentenceTransformer

print("Loading embedding model...")

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Embedding model loaded.")

## Step 5 — Load and Chunk Patient Records

### Why Chunking?

LLMs and embedding models both have token limits. If a document is too long, we can't embed it as one piece or fit it into the LLM's context. We split documents into **overlapping chunks**:

- **Chunk size** — how many characters per chunk (e.g. 600)
- **Overlap** — how many characters the next chunk shares with the previous one (e.g. 100)

Overlap ensures that sentences near a boundary aren't cut off and lose their surrounding context.

### What We Track Per Chunk
Each chunk stores:
- `text` — the raw content
- `source` — which file it came from (patient name)
- `chunk_id` — sequential index within the file

In [ ]:
import os

RECORDS_DIR = "./patient_records"
CHUNK_SIZE  = 600
CHUNK_OVERLAP = 100

def chunk_text(text, source, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append({
            "text": text[start:end].strip(),
            "source": source,
            "chunk_id": len(chunks)
        })
        start += chunk_size - overlap
    return chunks

all_chunks = []

for filename in os.listdir(RECORDS_DIR):
    if filename.endswith(".txt"):
        filepath = os.path.join(RECORDS_DIR, filename)
        with open(filepath, "r", encoding="utf-8") as f:
            content = f.read()
        file_chunks = chunk_text(content, source=filename)
        all_chunks.extend(file_chunks)
        print(f"📄 {filename} → {len(file_chunks)} chunks")

print(f"\nTotal chunks across all files: {len(all_chunks)}")

## Step 6 — Build the FAISS Vector Index

Now we embed every chunk and load them into a **FAISS index** for fast similarity search.

### How FAISS Works
FAISS (Facebook AI Similarity Search) stores all the chunk embeddings in a matrix. When you query it, it computes the **cosine similarity** (or inner product) between your query vector and all stored vectors and returns the top-k most similar ones in milliseconds — even with thousands of documents.

We use `IndexFlatIP` (inner product) with L2-normalized vectors, which is equivalent to cosine similarity. This is the simplest and most accurate FAISS index type.

In [ ]:
import faiss
import numpy as np

print("Embedding all chunks and building FAISS index...")

texts = [chunk["text"] for chunk in all_chunks]

embeddings = embedder.encode(texts, show_progress_bar=True, normalize_embeddings=True)

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(np.array(embeddings, dtype="float32"))

print(f"\nFAISS index built with {index.ntotal} vectors (dimension={dimension})")

## Step 7 — Define the Retrieval Function

The retriever takes a natural language question, embeds it using the same embedding model, and queries FAISS for the top-k most semantically similar chunks.

The returned chunks are sorted by similarity score. We return both the raw text and the source filename so the LLM (and user) know where the information came from.

In [ ]:
def retrieve(query, top_k=4):
    query_vec = embedder.encode([query], normalize_embeddings=True)
    scores, indices = index.search(np.array(query_vec, dtype="float32"), top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx != -1:
            results.append({
                "text": all_chunks[idx]["text"],
                "source": all_chunks[idx]["source"],
                "score": float(score)
            })
    return results

print("Retriever ready.")

## Step 8 — Define the Generation Function

This function assembles the **RAG prompt** and calls the LLM.

### Prompt Engineering

The prompt follows the **Gemma 2 instruction format** (`<start_of_turn>user ... <end_of_turn>`). The retrieved chunks are injected as a `CONTEXT` block before the user question. The system instructions tell the model to:

- Answer only from the provided context
- Cite which patient file the information comes from
- Say "I don't know" rather than hallucinate

This grounding is the core safety property of RAG over pure generation.

In [ ]:
def generate_answer(query, retrieved_chunks, max_tokens=512):
    context_blocks = []
    for i, chunk in enumerate(retrieved_chunks):
        context_blocks.append(f"[Source: {chunk['source']}]\n{chunk['text']}")
    context_str = "\n\n---\n\n".join(context_blocks)

    prompt = f"""<start_of_turn>user
You are a medical records assistant. Answer the question using ONLY the patient record context below.
Always mention which patient file your answer comes from.
If the answer is not found in the context, say: "I don't have that information in the provided records."

CONTEXT:
{context_str}

QUESTION: {query}
<end_of_turn>
<start_of_turn>model
"""

    response = llm(
        prompt,
        max_tokens=max_tokens,
        temperature=0.2,
        top_p=0.9,
        stop=["<end_of_turn>", "<start_of_turn>"]
    )

    return response["choices"][0]["text"].strip()

print("Generator ready.")

## Step 9 — Define the Full RAG Pipeline

The `ask()` function ties retrieval and generation together into a single user-facing interface:

1. Retrieve top-k relevant chunks for the query
2. Print which chunks were retrieved (for transparency)
3. Generate and return the LLM's grounded answer

The transparency step — showing which sources were used — is important in a medical context. It lets a clinician verify exactly where the model's answer came from.

In [ ]:
def ask(question, top_k=4, verbose_sources=True):
    print(f"\n{'='*60}")
    print(f"❓ Question: {question}")
    print(f"{'='*60}")

    chunks = retrieve(question, top_k=top_k)

    if verbose_sources:
        print("\n Retrieved Sources:")
        for i, c in enumerate(chunks):
            print(f"  [{i+1}] {c['source']}  (score: {c['score']:.3f})")

    print("\nGenerating answer...\n")
    answer = generate_answer(question, chunks)

    print(f"Answer:\n{answer}")
    print(f"{'='*60}\n")
    return answer

print("RAG pipeline ready. Call ask('your question') to query.")

## Step 10 — Query the RAG Chatbot

The system is now fully operational. Below are several example queries spanning different patients and question types — from medication lookups to cross-patient condition searches.

Each response will show:
- Which files were retrieved as context
- Their relevance scores (0.0–1.0, higher = more similar)
- The LLM's grounded answer with source attribution

> 💡 First inference may be slower as the model warms up. Subsequent queries are faster.

In [ ]:
ask("What medications was John Doe prescribed on his first visit?")

In [ ]:
ask("What is Maria Santos's date of birth and what condition is she being treated for?")

In [ ]:
ask("Which patients are on Metformin and why?")

In [ ]:
ask("Summarize Robert Kim's progress across his visits.")

In [ ]:
ask("Was John Doe ever referred to physical therapy, and if so, what were the results?")

In [ ]:
ask("Which patient has a cardiovascular risk management plan and what medications support it?")